# DATA 298B — Colab Live Dashboard Demo

This notebook prepares the existing React/Vite dashboard for a Colab runtime demo. It mounts the existing `DATA298B` Drive structure, clones `feature/298b-live-dashboard`, and starts the frontend shown in the final presentation screenshot.

The normal dashboard demo uses saved XGBoost and LLM report artifacts. The optional Gemma cell at the end performs one live adapter inference only when explicitly enabled.

In [ ]:
# Configuration — run this notebook from top to bottom.
REPO_URL = 'https://github.com/aryaMehta26/Graph-Based-Fraud-Intelligence-Platform.git'
REPO_BRANCH = 'feature/298b-live-dashboard'
DRIVE_ROOT = '/content/drive/MyDrive/DATA298B'
REPO_DIR = '/content/Graph-Based-Fraud-Intelligence-Platform'
FRONTEND_PORT = 4173
RUN_LIVE_GEMMA = True  # Required for the LLM Investigator live button.
GEMMA_ADAPTER_DIR = f'{DRIVE_ROOT}/adapters/gemma_full_grounded'
# If your Drive adapter has a different final name, change only this path.
GEMMA_MODEL_ID = ''  # Filled from adapter_config.json below; do not guess the base model.

In [ ]:
# Mount the existing shared Drive structure.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

repo_path = Path(REPO_DIR)
if repo_path.exists():
    subprocess.run(['git', '-C', str(repo_path), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
os.chdir(repo_path)
sys.path.insert(0, str(repo_path))
adapter_config_path = Path(GEMMA_ADAPTER_DIR) / 'adapter_config.json'
if adapter_config_path.exists():
    import json
    adapter_config = json.loads(adapter_config_path.read_text())
    discovered_model_id = adapter_config.get('base_model_name_or_path', '')
    if discovered_model_id:
        GEMMA_MODEL_ID = discovered_model_id
elif RUN_LIVE_GEMMA:
    raise FileNotFoundError(f'Gemma adapter_config.json not found: {adapter_config_path}')
os.environ['DATA298B_LIVE_GEMMA'] = '1' if RUN_LIVE_GEMMA else '0'
os.environ['DATA298B_GEMMA_MODEL_ID'] = GEMMA_MODEL_ID
os.environ['DATA298B_GEMMA_ADAPTER_DIR'] = GEMMA_ADAPTER_DIR
os.environ['DATA298B_GEMMA_TRACE_DIR'] = f'{DRIVE_ROOT}/artifacts/live_gemma_traces'
print('Repository:', repo_path)
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
live_commit = subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip()
api_source = (repo_path / 'scripts' / 'live_dashboard_api.py').read_text()
print('Live dashboard commit:', live_commit)
if '_recover_report_from_raw_response' not in api_source:
    raise RuntimeError('Stale dashboard checkout: raw Gemma report recovery is missing. Restart the runtime and rerun this notebook from the top.')
print('Gemma base model:', GEMMA_MODEL_ID or '<not discovered>')

In [ ]:
# Install the dashboard/runtime dependencies.
%pip install -q pandas pyarrow xgboost scikit-learn streamlit pyvis neo4j python-dotenv pyyaml transformers peft accelerate bitsandbytes sentencepiece

# Confirm the runtime and GPU. XGBoost/dashboard can run without a GPU; live Gemma cannot.
import torch
print('Torch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    if RUN_LIVE_GEMMA:
        raise RuntimeError('RUN_LIVE_GEMMA=True but this Colab runtime has no CUDA GPU. Change the runtime to a GPU before starting the API.')
    print('No CUDA GPU detected. Live Gemma is disabled; XGBoost replay is still available.')

In [ ]:
# Link the existing Drive input artifacts into the paths used by the repository snapshot tooling.
# This does not copy the multi-million-row Parquet files or modify the Drive data.
from pathlib import Path

drive_input = Path(DRIVE_ROOT) / 'input'
processed = Path(REPO_DIR) / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

def link_drive_file(source_name, target_name):
    source = drive_input / source_name
    target = processed / target_name
    if not source.exists():
        print('Missing:', source)
        return False
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(source)
    print(f'{target.name} -> {source}')
    return True

# The current Drive layout contains the graph-enriched train/test files.
link_drive_file('test_graph_enriched.parquet', 'test_graph_enriched.parquet')
link_drive_file('test_graph_enriched.parquet', 'split_test.parquet')
link_drive_file('train_graph_enriched.parquet', 'train_graph_enriched.parquet')

# Check the row count without loading the full Parquet file into RAM.
import pyarrow.parquet as pq
test_path = processed / 'test_graph_enriched.parquet'
if test_path.exists():
    print('Test rows:', pq.ParquetFile(test_path).metadata.num_rows)
    print('Columns:', pq.ParquetFile(test_path).schema.names[:20], '...')

In [ ]:
# Verify the repository and saved artifacts before starting the React dashboard.
from pathlib import Path
required = [
    Path(REPO_DIR) / 'frontend' / 'package.json',
    Path(REPO_DIR) / 'data/models/xgboost_baseline.json',
    Path(REPO_DIR) / 'data/models/xgboost_graph_enhanced.json',
    Path(REPO_DIR) / 'data/processed/test_graph_enriched.parquet',
]
for path in required:
    print(('OK  ' if path.exists() else 'MISS'), path)

print('Drive adapter path:', GEMMA_ADAPTER_DIR)
print('Adapter exists:', Path(GEMMA_ADAPTER_DIR).exists())
if RUN_LIVE_GEMMA and (not Path(GEMMA_ADAPTER_DIR).exists() or not GEMMA_MODEL_ID):
    raise RuntimeError('Live Gemma is enabled but the adapter path or base model ID is missing.')

In [ ]:
# Start the live Python bridge: replayed XGBoost predictions plus optional Gemma inference.
import os, subprocess, time, urllib.request, json
# Stop only a prior copy started by this notebook; do not kill unrelated Colab processes.
if 'api_process' in globals() and api_process.poll() is None:
    api_process.terminate()
    api_process.wait(timeout=10)
import socket
def port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(('127.0.0.1', port)) == 0
for port in (8765, FRONTEND_PORT):
    if port_is_open(port):
        raise RuntimeError(f'Port {port} is already occupied. Restart the Colab runtime, then run this notebook from the top.')
print('Ports 8765 and', FRONTEND_PORT, 'are available.')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
api_log_path = '/content/live_dashboard_api.log'
api_log = open(api_log_path, 'w')
api_process = subprocess.Popen(
    [sys.executable, 'scripts/live_dashboard_api.py'],
    cwd=REPO_DIR,
    stdout=api_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
for _ in range(30):
    if api_process.poll() is not None:
        print(Path(api_log_path).read_text()[-6000:])
        raise RuntimeError(f'Live API exited during startup with code {api_process.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8765/api/live/status', timeout=2) as response:
            status = json.loads(response.read())
        print('API health:', status)
        break
    except Exception:
        time.sleep(1)
else:
    print(Path(api_log_path).read_text()[-6000:])
    raise RuntimeError('Live API did not become healthy on port 8765.')
print('Live API PID:', api_process.pid)
print('API log:', Path(api_log_path).read_text()[-3000:])
with urllib.request.urlopen('http://127.0.0.1:8765/api/live/predictions?limit=1', timeout=5) as response:
    prediction_smoke = json.loads(response.read())
print('Prediction smoke test:', prediction_smoke)
probe_account = prediction_smoke['items'][0]['src_acct']
with urllib.request.urlopen(f'http://127.0.0.1:8765/api/graph/{probe_account}?limit=5', timeout=10) as response:
    graph_smoke = json.loads(response.read())
print('Network smoke test:', len(graph_smoke.get('edges', [])), 'edges for', probe_account)
print('Network route is served by the same live API at /api/graph/<account_id>.')

# Warm Gemma locally before exposing the dashboard through ngrok. This avoids a
# long first-request model load being terminated by the public gateway.
if RUN_LIVE_GEMMA:
    import threading
    warmup_result = {}
    def warmup_gemma():
        try:
            with urllib.request.urlopen('http://127.0.0.1:8765/api/live/warmup', timeout=1800) as response:
                warmup_result['payload'] = json.loads(response.read())
        except Exception as exc:
            warmup_result['error'] = exc
    warmup_thread = threading.Thread(target=warmup_gemma, daemon=True)
    warmup_thread.start()
    while warmup_thread.is_alive():
        lines = Path(api_log_path).read_text().splitlines()
        print('Gemma warm-up:', lines[-1] if lines else 'loading weights...', flush=True)
        time.sleep(10)
    if 'error' in warmup_result:
        print(Path(api_log_path).read_text()[-12000:])
        raise warmup_result['error']
    print('Gemma warm-up complete:', warmup_result['payload'])

In [ ]:
# Install and start the React/Vite dashboard shown in the final UI.
import subprocess, time
frontend_dir = Path(REPO_DIR) / 'frontend'
# Allow the temporary ngrok hostname used for the public demo URL.
vite_config = frontend_dir / 'vite.config.ts'
vite_text = vite_config.read_text()
if "allowedHosts:" not in vite_text:
    vite_text = vite_text.replace("server: {", "server: {\n      allowedHosts: ['.ngrok-free.dev', '.ngrok.io'],", 1)
    vite_config.write_text(vite_text)
print('Vite allowedHosts configured for Colab/ngrok.')
subprocess.run(['npm', 'install', '--silent'], cwd=frontend_dir, check=True)
log_path = '/content/frontend_dashboard.log'
log_file = open(log_path, 'w')
frontend_process = subprocess.Popen([
    'npm', 'run', 'dev', '--',
    '--host', '0.0.0.0',
    '--port', str(FRONTEND_PORT),
], stdout=log_file, stderr=subprocess.STDOUT, cwd=frontend_dir)
time.sleep(8)
if frontend_process.poll() is not None:
    print(Path(log_path).read_text()[-6000:])
    raise RuntimeError(f'Vite exited during startup with code {frontend_process.returncode}')
print('Frontend PID:', frontend_process.pid)
print('Log:', log_path)
print(Path(log_path).read_text()[-2000:])

In [ ]:
# Open the React dashboard inside the notebook using Colab's supported iframe helper.
from google.colab import output
output.serve_kernel_port_as_iframe(FRONTEND_PORT, height=900)

# Optional public link for teammates. Add a Colab Secret named NGROK_AUTHTOKEN first.
%pip install -q pyngrok
from google.colab import userdata
from pyngrok import ngrok

try:
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = ''

if ngrok_token:
    ngrok.kill()
    ngrok.set_auth_token(ngrok_token)
    public_url = ngrok.connect(FRONTEND_PORT, 'http').public_url
    print('Public dashboard URL:', public_url)
else:
    print('NGROK_AUTHTOKEN was not found in Colab Secrets; using the embedded iframe only.')

## Final live-stack health check

The React dashboard uses the checked-in DATA 298B snapshot for the reference pages, while the live API adds replayed XGBoost predictions and optional Gemma investigation requests.

In [ ]:
import urllib.request, json
with urllib.request.urlopen('http://127.0.0.1:8765/api/live/status', timeout=5) as response:
    print('Final API status:', json.loads(response.read()))
print('The live API and React UI are running.')
print('Overview polls live XGBoost predictions; LLM Investigator can call Gemma.')
print('Investigator includes transaction evidence, live graph metrics, and separate XGBoost/Gemma confidence labels.')

## Demo sequence

1. Keep `RUN_LIVE_GEMMA = True`, then run the setup cells on a GPU runtime.
2. Open **Overview** to show the live replayed XGBoost prediction stream.
3. Open **Transactions** and select a row with an XGBoost score of at least 0.300.
4. Click **Investigate live case**; the Investigator loads transaction and graph evidence.
5. Confirm the pre-inference value is labeled **XGBoost score**, then click **Run fresh Gemma report**.
6. Confirm the completed report changes the label to **Gemma confidence** and includes evidence/actions.
7. Open **Network** to inspect the same account neighborhood.
8. Use **Reports** only for the saved validated benchmark cases.

This is a Colab-hosted demo runtime. Closing or timing out the runtime stops the dashboard and any live model process.